<a href="https://colab.research.google.com/github/guyen5/Capstone/blob/main/notebooks/3%20Backtest%20Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 3: Backtest Engine

This notebook builds everything the backtest reads at each decision date. Wang and Hasuike retrain their model every month on 9 months of data, which is only 72 monthly rows for the eight ETF_A funds, so the replication trains on **daily rows**: every trading day's indicators paired with the return over the next 21 trading days. The notebook builds that daily table, the training windows that say which daily rows each test month may use, and the covariance matrices the optimizer needs. The optimizer, the fee and turnover accounting and the benchmarks are added to this notebook next (WBS 3.3).

**Work packages:** WBS 3.2 (Daily features and windows) and the inputs to 3.3 (Backtest engine)

**Inputs:** `Daily Prices.parquet` from Notebook 1 and `Monthly Data.parquet` from Notebook 2

**Outputs** (Google Drive, `Capstone/data`):

| File | Folder | Contents |
|---|---|---|
| Daily Features.parquet | processed | One row per universe, ETF and trading day: 7 features, the 21-day forward return and the day it becomes known |
| Daily Training Windows.parquet / .csv | processed | For each of the 81 test months and each window (9+3, 36, 60 months): which daily rows to fit, validate and predict on |
| Covariances.parquet | processed | Sample and Ledoit-Wolf covariance matrices at every month-end, both universes |
| Daily Data Checks.csv | reports | Results of all checks on the three tables |
| Variables.csv | processed | Definition of every column, now including the three new tables |

**Timing rule:** a daily row dated d holds features known at the close of d. Its target is the return from d to 21 trading days later, so the row can be used for training only from that later day (its `label_date`) onward. Section 4 checks this directly.

## 1. Setup

Connects Google Drive, where all data files are stored, and loads the project code from the GitHub repository. Run this cell first in every new Colab session. When Colab asks for access, choose the Google account that holds the `Capstone/data` folder.

In [ ]:
# Connect Google Drive. All data files are read from and saved to
# MyDrive/Capstone/data.
from google.colab import drive
drive.mount("/content/drive")

import os
import sys

REPO_URL = "https://github.com/guyen5/Capstone.git"
REPO_DIR = "/content/Capstone"

# Download the project code on the first run of a session; on later runs,
# update it so any changes pushed to GitHub are picked up.
if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull --quiet
else:
    !git clone --quiet {REPO_URL} {REPO_DIR}

# Install the pinned library versions listed in requirements.txt.
%pip install --quiet -r {REPO_DIR}/requirements.txt

# Make the project modules importable with "from src import ...".
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

In [ ]:
import importlib
import json
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import config, cleaning, features, windows, covariance, variables

# Reload the modules so code changes pulled from GitHub take effect
# without restarting the Colab runtime.
for module in (config, cleaning, features, windows, covariance, variables):
    importlib.reload(module)

DIRS = config.data_dirs()
pd.set_option("display.width", 160)

## 2. Load the inputs

Reads the clean daily prices from Notebook 1 and the monthly table from Notebook 2. The monthly table is used only to confirm that the daily table gives the same features at every month-end.

In [ ]:
clean = cleaning.load_clean(DIRS["processed"])
panel = pd.read_parquet(DIRS["processed"] / config.FILES["monthly"])
raw_info = json.loads((DIRS["raw"] / config.FILES["raw_info"]).read_text())

print(f"Daily prices: {len(clean):,} rows, {clean['ticker'].nunique()} ETFs, "
      f"{clean['date'].min():%Y-%m-%d} to {clean['date'].max():%Y-%m-%d}")
print(f"Monthly Data: {len(panel):,} rows, {panel['date'].nunique()} month-ends")
print("Raw file checksum:", raw_info["md5"])

## 3. Daily features table

The same seven indicators as the monthly table, now kept for every trading day from January 2010, together with the target.

| Column | Meaning |
|---|---|
| `month_end` | True on the last trading day of a month, the days on which the portfolio is rebalanced |
| seven features | Momentum, trend, price bias, RSI, MACD, Bollinger width, volume bias (as in Notebook 2) |
| `fwd_return` | Return over the next 21 trading days |
| `fwd_rel_return` | That return minus the average of the universe over the same days. This is the forecast target: with weights summing to one, only the differences between ETFs move the portfolio, and removing the common market move takes out the largest source of forecast error |
| `label_date` | The day the forward return becomes known |

Neighbouring rows overlap: two consecutive days share 20 of their 21 target days. The training windows in section 5 and the bootstrap in Notebook 5 are built around this. The last 21 trading days have features but no target yet.

In [ ]:
daily = features.daily_features(clean)
daily_table = features.build_daily_table(clean, daily)

print(daily_table.groupby("universe").agg(rows=("date", "size"),
                                         days=("date", "nunique"),
                                         month_ends=("month_end", "sum")))
print(f"Rows without a target yet: {daily_table['fwd_return'].isna().sum()} "
      f"(the last {config.FORWARD_DAYS} trading days)")
daily_table.head()

## 4. Checks on the daily table

Confirms the table is complete and that the target is correct:

- one row per ETF per trading day, no duplicates, no missing features;
- every target becomes known after its row date, and only the last 21 days lack one;
- relative returns sum to zero across each universe on each day;
- the forward return and its label date, recomputed independently from the price table for 200 random rows;
- at every month-end, the features equal those in the monthly table, which already passed the look-ahead test in Notebook 2.

Every column is also checked against the Variables list. The notebook stops on any FAIL.

In [ ]:
daily_checks = features.validate_daily(daily_table, clean, panel)
daily_columns = variables.check_table(daily_table, "Daily Features")

print(daily_checks.to_string(index=False))
print()
print(daily_columns.to_string(index=False))

checks = [daily_checks.assign(table="Daily Features"),
          daily_columns.assign(table="Daily Features")]
if (pd.concat(checks)["status"] == "FAIL").any():
    raise RuntimeError("A check failed. The daily table should not be used.")

## 5. Daily training windows

For each of the 81 test months, the window says which daily rows the model may use. A row is usable on the decision date only if its target is already known (`label_date` on or before the decision date). The window covers the most recent months of usable rows, and its last months are the validation slice used to tune settings.

| Window | Length | Validation | Used for |
|---|---|---|---|
| 9+3 | 12 months | last 3 months | The replication, as in Wang and Hasuike |
| 36 | 36 months | last 12 months | Candidate for the main test |
| 60 | 60 months | last 12 months | Candidate for the main test |

The main test uses one of the three, chosen on 2015 to 2019 data before any test result is seen (WBS 5.3).

**Gap before validation.** A training row from the last days before the validation slice would have a target that runs into the validation period, so tuning would see validation returns. Rows are therefore fitted only if their target is known before validation starts (`fit_end`), which leaves a gap of 21 trading days. After tuning, the model is refit on all rows up to `train_end`.

In [ ]:
all_daily_windows = {w: windows.daily_windows(daily_table, w) for w in config.DAILY_WINDOWS}
daily_window_table = pd.concat(all_daily_windows.values(), ignore_index=True)

for w, table in all_daily_windows.items():
    print(f"Window: {w}")
    report = windows.check_daily_windows(table, daily_table)
    print(report.to_string(index=False))
    print()
    checks.append(report.assign(table=f"Daily Training Windows {w}"))

window_columns = variables.check_table(daily_window_table, "Daily Training Windows")
checks.append(window_columns.assign(table="Daily Training Windows"))
print("Column checks:", window_columns["status"].value_counts().to_dict())

# Rows in the first test month (January 2020) for each window, ETF_A.
sizes = []
for w, table in all_daily_windows.items():
    fit, val, test = windows.daily_rows_for(daily_table, table.iloc[0], "ETF_A")
    sizes.append({"window": w, "fit rows": len(fit), "validation rows": len(val),
                  "prediction rows": len(test)})
print(pd.DataFrame(sizes).to_string(index=False))

all_daily_windows["9+3"].iloc[[0, 1, -1]]

## 6. Covariance matrices

At every month-end, the covariance of the ETFs' daily returns over the last 220 trading days, scaled to a monthly horizon (times 21). Two estimates are saved:

- **sample**: the plain sample covariance plus 10⁻⁶ on the diagonal, as in Wang and Hasuike. Used by the replication.
- **ledoit_wolf**: the sample covariance shrunk toward a scaled identity matrix, with the intensity chosen from the data (Ledoit and Wolf, 2004). Used by the robust model. SPY and VTI move almost together, so the sample matrix is close to singular; shrinkage keeps it well conditioned.

The checks confirm that every matrix is complete, symmetric and positive definite, that each window ends at its decision date, and that the sample estimate matches an independent calculation.

In [ ]:
month_ends = features.month_ends(clean)
month_ends = month_ends[month_ends >= pd.Timestamp(config.PANEL_START)]
cov_table = covariance.build_covariances(clean, month_ends)

cov_checks = covariance.validate_covariances(cov_table, clean)
cov_columns = variables.check_table(cov_table, "Covariances")
print(cov_checks.to_string(index=False))
print("Column checks:", cov_columns["status"].value_counts().to_dict())
checks += [cov_checks.assign(table="Covariances"), cov_columns.assign(table="Covariances")]

all_checks = pd.concat(checks, ignore_index=True)
if (all_checks["status"] == "FAIL").any():
    raise RuntimeError("A check failed. The covariance table should not be used.")

### 6a. How much the covariance is shrunk, and why

The left panel shows the Ledoit-Wolf shrinkage intensity at each month-end. The right panel shows the condition number of each matrix (largest eigenvalue divided by the smallest): a very large value means the optimizer can find near-riskless combinations that exist only because of estimation noise, such as long SPY and short VTI. Shrinkage brings it down by orders of magnitude.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
keys = cov_table.drop_duplicates(["date", "universe", "method"])
for uni in ("ETF_A", "ETF_B"):
    k = keys[(keys["universe"] == uni) & (keys["method"] == "ledoit_wolf")]
    axes[0].plot(k["date"], k["shrinkage"], label=uni)
axes[0].set_title("Ledoit-Wolf shrinkage intensity")
axes[0].set_ylabel("intensity (0 to 1)")
axes[0].legend()

cond = []
for (d, u, m), g in cov_table[cov_table["universe"] == "ETF_A"].groupby(["date", "universe", "method"]):
    ev = np.linalg.eigvalsh(g.pivot(index="row", columns="col", values="value").to_numpy())
    cond.append({"date": d, "method": m, "condition": ev.max() / ev.min()})
cond = pd.DataFrame(cond)
for m, g in cond.groupby("method"):
    axes[1].plot(g["date"], g["condition"], label=m)
axes[1].set_yscale("log")
axes[1].set_title("Condition number, ETF_A")
axes[1].legend()
plt.tight_layout()
plt.show()

print(cond.groupby("method")["condition"].median().round(0).to_string())

## 7. Save

Saves the three tables, the checks and an updated Variables list. An information file records the checksum of the raw download and the settings used.

In [ ]:
out = DIRS["processed"]
daily_table.to_parquet(out / config.FILES["daily_table"], index=False)
daily_window_table.to_parquet(out / config.FILES["daily_windows"], index=False)
daily_window_table.to_csv(out / "Daily Training Windows.csv", index=False)
cov_table.to_parquet(out / config.FILES["covariances"], index=False)
variables.to_frame().to_csv(out / config.FILES["variables"], index=False)
all_checks.to_csv(DIRS["reports"] / config.FILES["daily_checks"], index=False)

info = {
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "raw_file_md5": raw_info["md5"],
    "rows": {u: int(n) for u, n in daily_table["universe"].value_counts().items()},
    "trading_days": int(daily_table["date"].nunique()),
    "forward_days": config.FORWARD_DAYS,
    "windows": {k: list(v) for k, v in config.DAILY_WINDOWS.items()},
    "covariance_days": config.COV_DAYS,
    "covariance_ridge": config.COV_RIDGE,
    "test_months": [config.TEST_FIRST_MONTH, config.TEST_LAST_MONTH],
}
(out / config.FILES["daily_info"]).write_text(json.dumps(info, indent=2))
print("Saved: Daily Features, Daily Training Windows, Covariances, Variables, Daily Data Checks")

## 8. Summary

Key figures for the process log and the Data Specification sheet. After the run, save the notebook (File > Save) so the outputs are kept.

In [ ]:
passed = int((all_checks["status"] == "PASS").sum())
failed = int((all_checks["status"] == "FAIL").sum())
print(f"Trading days:   {daily_table['date'].nunique():,} "
      f"({daily_table['date'].min():%Y-%m-%d} to {daily_table['date'].max():%Y-%m-%d})")
for u, n in daily_table["universe"].value_counts().sort_index().items():
    print(f"Daily rows {u}: {n:,}")
print(f"Windows:        {', '.join(config.DAILY_WINDOWS)}; {len(all_daily_windows['9+3'])} test months each")
print(f"Covariances:    {cov_table[['date', 'universe', 'method']].drop_duplicates().shape[0]:,} matrices")
print(f"Checks:         {passed} passed, {failed} failed")